# Assignment 2: Agentic RAG & Simple Agent
## ReAct Agents, Custom Tools, and RAG with Agent Fallback

**OBJECTIVE:** Build custom tools, create a ReAct agent, implement a RAG pipeline, and combine them into an Agentic RAG system with fallback.

**LEARNING GOALS:**
- Create custom tools using the `@tool` decorator
- Build a ReAct agent with `create_react_agent`
- Implement a PDF-based RAG pipeline with FAISS
- Combine RAG with agent fallback for robust question answering

**INSTRUCTIONS:**
1. Complete each function by replacing the TODO comments with actual implementation
2. Run each cell after completing the function to test it
3. Each section builds on the previous one — work through them in order


In [ ]:
!pip install langchain_openai langchain_core langgraph langchain_huggingface langchain_community langchain_text_splitters langchain_classic faiss-cpu pypdf

In [1]:
import os
from getpass import getpass

os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API key: ")

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langgraph.prebuilt import create_react_agent
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0
)

/Users/sumisama/Desktop/sumit/outskill/ai-accelerator-C6/Day 12/day12/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/sumisama/Desktop/sumit/outskill/ai-accelerator-C6/Day 12/day12/lib/python3.11/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## 1. Custom Tools

Complete the two tool functions below using the `@tool` decorator.

**Concept:** Tools are Python functions decorated with `@tool` that an agent can call. The docstring tells the agent when and how to use the tool.

**Syntax reference:**
```python
@tool
def my_tool(input: str) -> str:
    """Description of what the tool does."""
    result = some_operation(input)
    return str(result)
```

**Why it matters:** Tools give agents the ability to interact with the outside world — run code, search the web, query databases, etc.

In [3]:
@tool
def calculator(expression: str) -> str:
    """
    Does math calculations. Pass a mathematical expression like '25 * 4'.

    TODO: Implement a safe math calculator tool.
    HINT: Validate that the expression only contains digits and operators (+-*/. ()), then use eval()

    Args:
        expression (str): A mathematical expression to evaluate

    Returns:
        str: The result of the calculation
    """
    # Define a set of allowed characters for safety
    allowed_chars = set("0123456789+-*/.() ")
    # Check if all characters in expression are allowed, then eval()
    if all(c in allowed_chars for c in expression):
        return str(eval(expression))
    else:
        raise ValueError("Invalid characters in expression")


@tool
def text_length(text: str) -> str:
    """
    Counts the length of the provided text in characters.

    TODO: Count text length in characters.
    HINT: Use len() and return a formatted string like 'Length: N characters'

    Args:
        text (str): The text to measure

    Returns:
        str: A formatted string with the character count
    """
    # Use len() on the text and return a formatted string
    return f"Length: {len(text)} characters"


tools = [calculator, text_length]

# Test the tools directly
print(f"Calculator test: 25 * 4 = {calculator.invoke('25 * 4')}")
print(f"Text length test: {text_length.invoke('Hello World')}")

Calculator test: 25 * 4 = 100
Text length test: Length: 11 characters


## 2. ReAct Agent

Complete the two functions below to create and run a ReAct agent.

**Concept:** A ReAct agent follows a Reason-Act loop: it thinks about which tool to use, calls it, reasons about the result, and repeats until it has a final answer.

**Syntax reference:**
```python
agent = create_react_agent(llm, tools)
response = agent.invoke({"messages": [HumanMessage(content="...")]})
```

**Why it matters:** Agents can autonomously decide which tools to call and how to combine their results to answer complex questions.

In [4]:
def create_agent(llm, tools):
    """
    Create a ReAct agent from an LLM and a list of tools.

    TODO: Create a ReAct agent.
    HINT: Use create_react_agent(llm, tools) from langgraph.prebuilt

    Args:
        llm: The language model to use
        tools: List of tool functions

    Returns:
        The created ReAct agent
    """
    agent = create_react_agent(llm, tools)
    return agent


def run_agent(agent, query: str) -> str:
    """
    Run a query through the agent and return the final AI response.

    TODO: Invoke the agent and extract the final AIMessage content.
    HINT: Use agent.invoke() with {"messages": [HumanMessage(content=query)]}.
          Then iterate reversed(response["messages"]) to find the last AIMessage with content.

    Args:
        agent: The ReAct agent
        query (str): The user's question

    Returns:
        str: The agent's final response
    """
    # Invoke the agent with a HumanMessage
    response = agent.invoke({"messages": [HumanMessage(content=query)]})
    # Loop through reversed messages to find the last AIMessage with content
    for msg in reversed(response["messages"]):
        if isinstance(msg, AIMessage) and msg.content:
            return msg.content
    return "No response"


# Test the agent
agent = create_agent(llm, tools)
print("Agent created:", agent is not None)

print("\n=== Test 1: Calculator ===")
print(run_agent(agent, "What is 25 * 4?"))

print("\n=== Test 2: Text Length ===")
print(run_agent(agent, "How long is the text 'Hello World'?"))

print("\n=== Test 3: General Knowledge ===")
print(run_agent(agent, "Who is Mark Zuckerberg?"))

/var/folders/00/r4pz_0hx17jd1xt8sdlp2xl00000gq/T/ipykernel_65789/4236590772.py:15: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


Agent created: True

=== Test 1: Calculator ===
The result of \( 25 \times 4 \) is 100.

=== Test 2: Text Length ===
The text 'Hello World' is 11 characters long.

=== Test 3: General Knowledge ===
Mark Zuckerberg is an American computer programmer and internet entrepreneur best known for co-founding Facebook, Inc. (now Meta Platforms, Inc.), a social media platform that has become one of the largest and most influential technology companies in the world. Born on May 14, 1984, in White Plains, New York, he developed an interest in programming at a young age.

Zuckerberg launched Facebook in 2004 while he was a student at Harvard University, initially as a social networking site for college students. The platform quickly expanded to include users from other universities and eventually the general public. Under his leadership, Facebook grew rapidly, introducing features like the News Feed, the "Like" button, and various advertising models.

In addition to his role at Facebook, Zuckerberg

## 3. PDF RAG Pipeline

Complete the function below to build a RAG pipeline from a PDF file.

**Concept:** RAG (Retrieval-Augmented Generation) loads documents, splits them into chunks, embeds them into a vector store, and retrieves relevant chunks to answer questions.

**Syntax reference:**
```python
docs   = PyPDFLoader(path).load()
chunks = RecursiveCharacterTextSplitter(...).split_documents(docs)
db     = FAISS.from_documents(chunks, embeddings)
retriever = db.as_retriever(search_kwargs={"k": ...})
```

**Why it matters:** RAG lets you ground LLM responses in your own documents, reducing hallucination and enabling domain-specific Q&A.

In [5]:
def build_rag_pipeline(pdf_path: str):
    """
    Load a PDF, chunk it, embed with HuggingFace, store in FAISS, and return a retriever.

    TODO: Implement the full RAG ingestion pipeline.
    HINT: PyPDFLoader -> RecursiveCharacterTextSplitter -> HuggingFaceEmbeddings -> FAISS -> retriever

    Args:
        pdf_path (str): Path to the PDF file

    Returns:
        A FAISS retriever object
    """
    # Load the PDF
    docs = PyPDFLoader(pdf_path).load()

    # Split into chunks (chunk_size=1000, chunk_overlap=200)
    chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)

    # Create embeddings using HuggingFace (model: "sentence-transformers/all-MiniLM-L6-v2")
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    # Build FAISS vector store from chunks
    db = FAISS.from_documents(chunks, embeddings)

    # Return a retriever (k=5)
    retriever = db.as_retriever(search_kwargs={"k": 5})
    return retriever


# Test the function using the provided study guide PDF
retriever = build_rag_pipeline("rag_study_guide.pdf")
print(f"Retriever created: {retriever is not None}")
# print("Uncomment the lines above to test. A sample PDF (rag_study_guide.pdf) is provided in this folder.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8988.48it/s]


Retriever created: True


## 4. Agentic RAG with Fallback

Complete the function below to combine RAG with an agent fallback.

**Concept:** Try answering from your documents first (RAG). If the documents don't contain enough information, fall back to the agent which can use tools and its own knowledge.

**Syntax reference:**
```python
document_chain  = create_stuff_documents_chain(llm, prompt)
retrieval_chain = create_retrieval_chain(retriever, document_chain)
result          = retrieval_chain.invoke({"input": query})
answer          = result["answer"]
```

**Why it matters:** This pattern gives you the best of both worlds — grounded answers from your data, with a safety net for questions outside your corpus.

In [6]:
def agentic_rag_query(query: str, retriever, agent) -> dict:
    """
    Try RAG first; if the answer is INSUFFICIENT_CONTEXT, fall back to the agent.

    TODO: Build a retrieval chain, check if response is insufficient, and fall back to agent.
    HINT: Create a prompt that says "respond with INSUFFICIENT_CONTEXT if context doesn't help".
          Use create_stuff_documents_chain + create_retrieval_chain.
          If the answer contains 'INSUFFICIENT_CONTEXT', call run_agent instead.

    Args:
        query (str): The user's question
        retriever: A FAISS retriever from build_rag_pipeline
        agent: A ReAct agent from create_agent

    Returns:
        dict: Contains 'answer' and 'source' keys
    """
    # Create a RAG prompt (instruct LLM to say INSUFFICIENT_CONTEXT if context doesn't help)
    rag_prompt = ChatPromptTemplate.from_template(
        template="Use the retrieved documents context to answer the question: {input} \ncontext: {context}.\n If the documents don't help, respond with 'INSUFFICIENT_CONTEXT'."
    )

    # Build the retrieval chain
    document_chain = create_stuff_documents_chain(llm, rag_prompt)
    retrieval_chain = create_retrieval_chain(retriever, document_chain)

    # Invoke and get the answer
    rag_result = retrieval_chain.invoke({"input": query})
    rag_answer = rag_result["answer"]

    # If RAG answer is good, return it with source "PDF Documents"
    # Otherwise fall back to run_agent and return with source "Agent"
    if "INSUFFICIENT_CONTEXT" in rag_answer:
        agent_answer = run_agent(agent, query)
        return {"answer": agent_answer, "source": "Agent - not in the PDF"}
    return {"answer": rag_answer, "source": "PDF Documents"}

# Test the function (requires a retriever from Section 3)
retriever = build_rag_pipeline("rag_study_guide.pdf")
result = agentic_rag_query("What is RAG?", retriever, agent)
print(f"Source: {result['source']}")
print(f"Answer: {result['answer']}")

result2 = agentic_rag_query("Who is the current US president?", retriever, agent)
print(f"Source: {result2['source']}")   # Should say "Agent" — not in the PDF
print(f"Answer: {result2['answer']}")
# print("Uncomment the lines above after completing build_rag_pipeline to test agentic_rag_query.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8077.73it/s]


Source: PDF Documents
Answer: RAG stands for Retrieval-Augmented Generation. It is a technique that enhances the responses of large language models (LLMs) by grounding them in external documents. Instead of solely relying on the model's training data, RAG retrieves the most relevant passages from a document store and incorporates them into the prompt. This approach allows the model to answer questions about documents it has not encountered during training, resulting in lower rates of hallucination compared to open-ended generation. A typical RAG pipeline involves stages such as loading documents, splitting them into smaller chunks, embedding those chunks into vectors, storing them in a vector database, and retrieving the most similar chunks at query time to provide context for the LLM's responses.
Source: Agent - not in the PDF
Answer: As of my last knowledge update in October 2023, the current President of the United States is Joe Biden. He has been in office since January 20, 2021. P

## 5. Final Test - Complete Pipeline

Once you've completed all the functions above, run this cell to test the complete pipeline.

In [7]:
print("Testing Complete Agentic RAG Pipeline")
print("=" * 50)

# Step 1: Tools
print("\nStep 1: Custom Tools...")
calc_result = calculator.invoke("10 + 20 * 3")
len_result = text_length.invoke("Hello World")
step1_ok = calc_result == "70" and "11" in len_result
print(f"   Calculator: 10 + 20 * 3 = {calc_result}")
print(f"   Text Length: {len_result}")

# Step 2: Agent
print("\nStep 2: ReAct Agent...")
test_agent = create_agent(llm, tools)
step2_ok = test_agent is not None
if step2_ok:
    agent_response = run_agent(test_agent, "What is 15 * 8?")
    step2_ok = agent_response is not None and len(agent_response) > 0
    print(f"   Agent response: {agent_response[:80]}")
else:
    print("   Failed to create agent")

# Step 3: RAG Pipeline (check function exists)
print("\nStep 3: RAG Pipeline...")
step3_ok = callable(build_rag_pipeline)
print(f"   build_rag_pipeline is callable: {step3_ok}")
print("   (Provide a PDF path to fully test this function)")

# Step 4: Agentic RAG (check function exists)
print("\nStep 4: Agentic RAG with Fallback...")
step4_ok = callable(agentic_rag_query)
print(f"   agentic_rag_query is callable: {step4_ok}")
print("   (Provide a retriever to fully test this function)")

# Summary
print("\n" + "=" * 50)
print("Assignment Status:")
print(f"   Custom Tools: {'Pass' if step1_ok else 'Fail'}")
print(f"   ReAct Agent: {'Pass' if step2_ok else 'Fail'}")
print(f"   RAG Pipeline: {'Pass' if step3_ok else 'Fail'}")
print(f"   Agentic RAG: {'Pass' if step4_ok else 'Fail'}")

if all([step1_ok, step2_ok, step3_ok, step4_ok]):
    print("\nCongratulations! You've successfully completed the assignment!")
    print("You've built a complete Agentic RAG system!")
else:
    print("\nPlease complete the TODO functions above to finish the assignment.")

Testing Complete Agentic RAG Pipeline

Step 1: Custom Tools...
   Calculator: 10 + 20 * 3 = 70
   Text Length: Length: 11 characters

Step 2: ReAct Agent...


/var/folders/00/r4pz_0hx17jd1xt8sdlp2xl00000gq/T/ipykernel_65789/4236590772.py:15: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


   Agent response: The result of \( 15 \times 8 \) is 120.

Step 3: RAG Pipeline...
   build_rag_pipeline is callable: True
   (Provide a PDF path to fully test this function)

Step 4: Agentic RAG with Fallback...
   agentic_rag_query is callable: True
   (Provide a retriever to fully test this function)

Assignment Status:
   Custom Tools: Pass
   ReAct Agent: Pass
   RAG Pipeline: Pass
   Agentic RAG: Pass

Congratulations! You've successfully completed the assignment!
You've built a complete Agentic RAG system!
